<a href="https://colab.research.google.com/github/ErVijayRaghuwanshi/GenAI/blob/main/deep_agents_101_template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Agents 101: Template

This is a blank version of the Deep Agents 101 workshop: same steps (harness, system prompt, tools), no persona or use case baked in. Use it as a starting point to build your own agent.

For topics not covered today, see the self-paced [LangChain Academy Deep Agents course.](https://academy.langchain.com/courses/foundation-introduction-to-deepagents)

In [ ]:
# Wraps text to make it easier to read

from IPython.display import HTML, display

def set_css():
    display(HTML('''
    <style>
      pre { white-space: pre-wrap; word-break: break-word; }
    </style>
    '''))

get_ipython().events.register('pre_run_cell', set_css)

## Setup: connect a model

**What you'll do:** install the SDKs and provide a model key.

In [9]:
%pip install -q deepagents langgraph langchain-nvidia-ai-endpoints langchain-openai langchain-anthropic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.9/327.9 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.7/163.7 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 787.6/787.6 kB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4

In [5]:
# from google.colab import userdata

# GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
# MODEL = userdata.get("MODEL")
# To use an OpenAI key (or another model) instead, just switch it:
# OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
import os

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
MODEL = os.getenv("MODEL") or "google_genai:gemini-3.1-flash-lite"  # default to Gemini 3.6 Flash Lite if not set


In [6]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model=MODEL,
    api_key=GEMINI_API_KEY,
)

# To use a different provider instead, replace the block above, for example:
# model = init_chat_model("openai:gpt-6", api_key=OPENAI_API_KEY)

## 1: Invoking The Agent

In [7]:
from deepagents import create_deep_agent

agent = create_deep_agent(model=model,
                          debug=True)

# Write a message here
message = "Hello World"

result = agent.invoke({"messages": [{"role": "user", "content": message}]})

for m in result['messages']:
    m.pretty_print()

[values] {'messages': [HumanMessage(content='Hello World', additional_kwargs={}, response_metadata={}, id='83d2ad30-6557-4f87-857a-fb65617859f0')], 'files': {}}
[updates] {'PatchToolCallsMiddleware.before_agent': None}
[updates] {'model': {'messages': [AIMessage(content=[{'type': 'text', 'text': 'Hello! How can I help you today?', 'extras': {'signature': 'EnEKbwFpFH0TiPTDq0S8rh11MQn9H1qRmrPhaPcaBbuOlbF1O+7M2EMH9uJs/3zQUnKjpWozN+jeXOhrWrmJfaXfUGjPL42yPQBreXl/hQet6y+jvmu9E8ZlsgANdbu6neP8Fi/fPFhJNPa4cDdUS2/vAQ=='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0c4fa-8ff7-7552-9c01-736bdaf730b1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2187, 'output_tokens': 9, 'total_tokens': 2196, 'input_token_details': {'cache_read': 0}})]}}
[values] {'messages': [HumanMessage(content='Hello World', additional_kwargs={}, response_metadata={}

## 2: Set its role with a system prompt

In [ ]:
system_prompt = ""

# Examples:
# system_prompt = "You are a pirate. Answer only in pirate speak."
# system_prompt = "You are a research agent. Investigate a topic thoroughly and summarize the key findings with sources."
# system_prompt = "You are a support agent for a cooking app. Only answer questions about cooking, recipes, or the app's features. Politely decline anything else and redirect back to those topics."

agent = create_deep_agent(model=model, system_prompt=system_prompt)

# Replace the ___ placeholder below with your own content
message = "Write three sentences about ___."

result = agent.invoke({"messages": [{"role": "user", "content": message}]})

for m in result['messages']:
    m.pretty_print()

## 3: Give it a custom tool

**Anatomy of a tool**: `@tool` turns a plain function into something the model can call:
- the **docstring** becomes the tool's description (how the model decides when to use it)
- the **type hints** become its input schema (what arguments it expects)
- the **return value** becomes what the model sees back

**The tools we have provided below:**

*   **roll_dice**: rolls one or more dice (defaults to a single 6-sided die) and returns the results.
*   **password_generator**: generates a random password of a given length.
*   **random_joke**: fetches a random joke from a public API.
*   **random_fact**: fetches a random useless-but-true fact.
*   **currency_convert**: converts an amount between currencies using current exchange rates.
*   **wikipedia_summary**: fetches a summary from wikipedia about a topic.
*   **weather_lookup**: gets the current weather for a city
*   **stock_lookup**: gets the current information for a stock ticker
*   **hackernews_top_stories**: Get the top stories on Hacker News with titles and links.
*   **pokemon_lookup**: looks up a Pokemon's height, weight, types, and abilities by name.
*   **random_pokemon**: fetches a random Pokemon's height, weight, types, and abilities.

In [ ]:
from langchain.tools import tool
import os
import requests
import random
import secrets
import string

@tool
def roll_dice(sides: int = 6, count: int = 1) -> str:
    """Roll `count` dice with `sides` sides each."""
    rolls = [random.randint(1, sides) for _ in range(count)]
    return f"Rolled: {rolls} (total: {sum(rolls)})"

@tool
def password_generator(length: int = 12) -> str:
    """Generate a random password of the given length."""
    alphabet = string.ascii_letters + string.digits + "!@#$%^&*"
    return "".join(secrets.choice(alphabet) for _ in range(length))

@tool
def random_joke() -> str:
    """Fetch a random joke."""
    joke = requests.get("https://official-joke-api.appspot.com/random_joke").json()
    return f"{joke['setup']} ... {joke['punchline']}"

@tool
def currency_convert(amount: float, from_currency: str, to_currency: str) -> str:
    """Convert an amount from one currency to another using current exchange rates."""
    r = requests.get(
        "https://api.frankfurter.app/latest",
        params={"amount": amount, "from": from_currency.upper(), "to": to_currency.upper()},
    ).json()
    rate = r["rates"].get(to_currency.upper())
    if rate is None:
        return f"Couldn't convert {from_currency} to {to_currency}."
    return f"{amount} {from_currency.upper()} = {rate:.2f} {to_currency.upper()}"

@tool
def random_fact() -> str:
    """Fetch a random useless-but-true fact."""
    fact = requests.get("https://uselessfacts.jsph.pl/api/v2/facts/random", params={"language": "en"}).json()
    return fact["text"]

@tool
def wikipedia_summary(topic: str) -> str:
    """Get a short encyclopedia summary of a topic."""
    import requests
    r = requests.get(f"https://en.wikipedia.org/api/rest_v1/page/summary/{topic}")
    return r.text

@tool
def weather_lookup(city: str) -> str:
    """Look up the current weather for a city."""
    import requests

    r = requests.get(f"https://wttr.in/{city}", params={"format": "j1"}).json()
    return str(r["current_condition"][0])

@tool
def stock_lookup(symbol: str) -> str:
    """Look up current information about a stock by ticker symbol."""
    import requests

    r = requests.get(
        f"https://query1.finance.yahoo.com/v8/finance/chart/{symbol.upper()}",
        headers={"User-Agent": "Mozilla/5.0"},
    )
    return r.text

@tool
def hackernews_top_stories(count: int = 10) -> str:
    """Get the top stories on Hacker News with titles and links."""
    import requests
    r = requests.get("https://hn.algolia.com/api/v1/search", params={"tags": "front_page", "hitsPerPage": count})
    return r.text

@tool
def pokemon_lookup(name: str) -> str:
    """Look up a Pokemon's height, weight, types, and abilities by name."""
    r = requests.get(f"https://pokeapi.co/api/v2/pokemon/{name.lower()}")
    if r.status_code != 200:
        return f"Couldn't find a Pokemon named '{name}'."
    data = r.json()
    types = ", ".join(t["type"]["name"] for t in data["types"])
    abilities = ", ".join(a["ability"]["name"] for a in data["abilities"])
    return (f"{data['name'].title()}: height {data['height']}, weight {data['weight']}, "
            f"type(s) {types}, abilities: {abilities}")

@tool
def random_pokemon() -> str:
    """Fetch a random Pokemon's height, weight, types, and abilities."""
    dex_number = random.randint(1, 1025)
    data = requests.get(f"https://pokeapi.co/api/v2/pokemon/{dex_number}").json()
    types = ", ".join(t["type"]["name"] for t in data["types"])
    abilities = ", ".join(a["ability"]["name"] for a in data["abilities"])
    return (f"{data['name'].title()}: height {data['height']}, weight {data['weight']}, "
            f"type(s) {types}, abilities: {abilities}")

To write your own tool instead of using the examples above, add it below.

In [ ]:
# Define your system prompt
system_prompt = "You are an agent"

# Pick a tool, or write your own @tool function
TOOLS = []

agent = create_deep_agent(
    model=model,
    system_prompt=system_prompt,
    tools=TOOLS
    )

# Invoke agent
message = "Hello"

result = agent.invoke({"messages": [{"role": "user", "content": message}]})

for m in result['messages']:
    m.pretty_print()

## OPTIONAL: Include short-term memory so you can respond to your agent

Short-term memory with any agent built using any LangChain agent framework, including Deep Agents, is managed using a **checkpointer**.

In [ ]:
# Import the checkpointer

from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

In [ ]:
# The following thread_id is what tracks the conversation over turns, change the thread_id to change the conversation
config = {"configurable": {"thread_id": "1"}}

In [ ]:
# Define agent from above with the checkpointer
agent = create_deep_agent(
    model=model,
    system_prompt=system_prompt,
    tools=TOOLS,
    checkpointer=checkpointer
    )

# Invoke agent with the thread_id to track the conversation between turns
message = "Hello my name is Jess"

result = agent.invoke({"messages": [{"role": "user", "content": message}]},
                      config=config
                      )

for m in result['messages']:
    m.pretty_print()

In [ ]:
# Send a follow up message
message = "What's my name?"

result = agent.invoke({"messages": [{"role": "user", "content": message}]},
                      config=config
                      )

for m in result['messages']:
    m.pretty_print()

## OPTIONAL: Human-in-the-loop, approve/reject a risky action before it happens

**What this covers:** how `interrupt_on` pauses an agent mid-run so a human can approve, edit, or reject a specific tool call before it executes.

**The payoff:** any agent with access to money, message sends, or irreversible actions needs this kind of control before it is used in production.

Every tool call passes through an `Interrupt?` check. If a call matches a rule configured in `interrupt_on`, the agent pauses and hands control to a human instead of executing it directly. The human can approve the call as written, edit its arguments before it runs, or reject it outright, and the agent resumes from exactly where it paused.

This pause only works because a `checkpointer` is attached to the agent: it saves the agent's state (its messages, files, and progress so far) at the interrupt point so the run can be resumed later, potentially after the human has stepped away and come back. Resuming looks like calling `agent.invoke` again with `Command(resume={"decisions": [{"type": "approve"}]})` (or `"edit"` / `"reject"`), rather than starting a new conversation from scratch.

### Try it yourself

Build the pause described above, using the tool you picked in section 3. Gate it with `interrupt_on`, then resume the run with an approval decision.

In [ ]:
from langgraph.types import Command

# Define which tools you want to interrupt on in the following dic or leave as is to interrupt on the first tool in the tool list
interrupt_config = {tools[0]: True}

agent = create_deep_agent(
    model=model,
    system_prompt=system_prompt,
    tools=TOOLS,
    interrupt_on=interrupt_config,
    checkpointer=checkpointer,
)

config = {"configurable": {"thread_id": "hitl-demo"}}

# Reuse (or adjust) the prompt from section 3, it should trigger the same tool call
result = agent.invoke(
    {"messages": [{"role": "user", "content":
        "Use your tool to look up Pikachu, then tell me what you found."
    }]},
    config=config,
)

if "__interrupt__" in result:
    request = result["__interrupt__"][0].value
    print("Paused for approval:")
    for action in request["action_requests"]:
        print(f"  {action['name']}({action['args']})")
else:
    print(result["messages"][-1].content)

The cell above should pause instead of finishing, because your tool matched `interrupt_on`. Run the cell below to resume it with an approval decision.

In [ ]:
# Other decisions you could pass here instead of "approve":
#   {"type": "reject", "message": "..."}  # skip the tool call; `message` (optional) tells the agent why, so it doesn't just retry
#   {"type": "edit", "edited_action": {"name": "pokemon_lookup", "args": {"name": "charmander"}}}  # run the tool, but with different args than the model chose
result = agent.invoke(Command(resume={"decisions": [{"type": "approve"}]}), config=config)
print(result["messages"][-1].content)

## Wrap-up

You built: a filesystem-backed agent, a way to swap personas, and a custom tool.

Also covered: human-in-the-loop gating, added with a single argument.

This is a blank template: swap in your own system prompt, starter prompt, and tools to build a completely different agent.

Not covered today, but in the full LangChain Academy Deep Agents course: subagent delegation, backends (filesystem/store/composite), skills, memory across sessions, sandboxes, and deployment.